In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

# Reproducible results
np.random.seed(42)

# Simulated customer-product purchase data
data = {
    "Customer": [
        "C1", "C1", "C1",
        "C2", "C2", "C2",
        "C3", "C3",
        "C4", "C4", "C4",
        "C5", "C5",
        "C6", "C6", "C6"
    ],
    "Product": [
        "Laptop", "Mouse", "Keyboard",
        "Laptop", "Mouse", "Monitor",
        "Phone", "Earphones",
        "Laptop", "Keyboard", "Monitor",
        "Phone", "Earphones",
        "Laptop", "Mouse", "Monitor"
    ],
    "Rating": [
        5, 4, 5,
        4, 5, 4,
        5, 4,
        5, 4, 5,
        4, 5,
        4, 4, 5
    ]
}

df = pd.DataFrame(data)

print("Dataset Shape:", df.shape)
display(df)

Dataset Shape: (16, 3)


,Customer,Product,Rating
0,C1,Laptop,5
1,C1,Mouse,4
2,C1,Keyboard,5
3,C2,Laptop,4
4,C2,Mouse,5
5,C2,Monitor,4
6,C3,Phone,5
7,C3,Earphones,4
8,C4,Laptop,5
9,C4,Keyboard,4


In [2]:
# Convert transaction data into a customer-product matrix
user_item_matrix = df.pivot_table(
    index="Customer",
    columns="Product",
    values="Rating"
).fillna(0)

print("Customer-Product Matrix:")
display(user_item_matrix)

Customer-Product Matrix:


Product,Earphones,Keyboard,Laptop,Monitor,Mouse,Phone
Customer,,,,,,
C1,0.0,5.0,5.0,0.0,4.0,0.0
C2,0.0,0.0,4.0,4.0,5.0,0.0
C3,4.0,0.0,0.0,0.0,0.0,5.0
C4,0.0,4.0,5.0,5.0,0.0,0.0
C5,5.0,0.0,0.0,0.0,0.0,4.0
C6,0.0,0.0,4.0,5.0,4.0,0.0


In [3]:
# Calculate cosine similarity between customers
similarity_matrix = cosine_similarity(user_item_matrix)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("Customer Similarity Scores:")
display(similarity_df.round(2))

Customer Similarity Scores:


Customer,C1,C2,C3,C4,C5,C6
Customer,,,,,,
C1,1.00,0.65,0.00,0.68,0.00,0.59
C2,0.65,1.00,0.00,0.65,0.00,0.98
C3,0.00,0.00,1.00,0.00,0.98,0.00
C4,0.68,0.65,0.00,1.00,0.00,0.73
C5,0.00,0.00,0.98,0.00,1.00,0.00
C6,0.59,0.98,0.00,0.73,0.00,1.00


In [4]:
def recommend_products(customer_id, top_n=3):

    # Find customers most similar to the selected customer
    similar_customers = similarity_df[customer_id].drop(
        customer_id
    ).sort_values(ascending=False)

    # Products already rated by the customer
    purchased_products = user_item_matrix.loc[customer_id]
    purchased_products = purchased_products[
        purchased_products > 0
    ].index.tolist()

    # Calculate recommendation scores
    scores = {}

    for other_customer, similarity in similar_customers.items():

        for product in user_item_matrix.columns:

            rating = user_item_matrix.loc[
                other_customer, product
            ]

            if rating > 0 and product not in purchased_products:

                scores[product] = (
                    scores.get(product, 0)
                    + similarity * rating
                )

    # Sort products by recommendation score
    recommendations = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    return pd.DataFrame(
        recommendations,
        columns=["Recommended_Product", "Recommendation_Score"]
    )


# Example: Recommend products for C1
recommend_products("C1")

,Recommended_Product,Recommendation_Score
0,Monitor,8.952405
1,Earphones,0.000000
2,Phone,0.000000


In [5]:
def recommend_products(customer_id, top_n=3):

    similar_customers = similarity_df[customer_id].drop(
        customer_id
    ).sort_values(ascending=False)

    purchased_products = user_item_matrix.loc[customer_id]
    purchased_products = purchased_products[
        purchased_products > 0
    ].index.tolist()

    scores = {}

    for other_customer, similarity in similar_customers.items():

        for product in user_item_matrix.columns:

            rating = user_item_matrix.loc[
                other_customer, product
            ]

            if rating > 0 and product not in purchased_products:

                scores[product] = (
                    scores.get(product, 0)
                    + similarity * rating
                )

    # Keep only products with positive scores
    scores = {
        product: score
        for product, score in scores.items()
        if score > 0
    }

    recommendations = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    return pd.DataFrame(
        recommendations,
        columns=["Recommended_Product", "Recommendation_Score"]
    )


recommend_products("C1")

,Recommended_Product,Recommendation_Score
0,Monitor,8.952405


In [7]:
all_recommendations = []

for customer in user_item_matrix.index:

    recs = recommend_products(customer, top_n=3)

    for _, row in recs.iterrows():
        all_recommendations.append({
            "Customer": customer,
            "Recommended_Product": row["Recommended_Product"],
            "Recommendation_Score": round(
                row["Recommendation_Score"], 2
            )
        })

recommendation_df = pd.DataFrame(all_recommendations)

print("Personalized Recommendations:")
display(recommendation_df)

Personalized Recommendations:


,Customer,Recommended_Product,Recommendation_Score
0,C1,Monitor,8.95
1,C2,Keyboard,5.87
2,C4,Mouse,8.92
3,C6,Keyboard,5.87


In [8]:
# Simple recommendation relevance evaluation

test_customer = "C1"
test_product = "Monitor"

# Check whether the product is already rated
print("Customer:", test_customer)
print("Hidden product for testing:", test_product)

# Generate recommendations
test_recommendations = recommend_products(
    test_customer,
    top_n=3
)

display(test_recommendations)

# Check whether the hidden product appears
if test_product in test_recommendations[
    "Recommended_Product"
].values:
    print("Result: Relevant recommendation found!")
else:
    print("Result: Hidden product was not recommended.")

Customer: C1
Hidden product for testing: Monitor


,Recommended_Product,Recommendation_Score
0,Monitor,8.952405


Result: Relevant recommendation found!


In [9]:
# Save the original data
original_matrix = user_item_matrix.copy()
original_similarity = similarity_df.copy()

test_customer = "C1"
test_product = "Monitor"

# Temporarily hide the product rating
user_item_matrix.loc[test_customer, test_product] = 0

# Recalculate similarities without the hidden rating
similarity_matrix = cosine_similarity(user_item_matrix)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

# Generate recommendations without seeing the hidden rating
test_recommendations = recommend_products(
    test_customer,
    top_n=3
)

display(test_recommendations)

if test_product in test_recommendations["Recommended_Product"].values:
    print("Relevant recommendation found!")
else:
    print("Hidden product was not recommended.")

# Restore the original data
user_item_matrix = original_matrix
similarity_df = original_similarity

,Recommended_Product,Recommendation_Score
0,Monitor,8.952405


Relevant recommendation found!


In [10]:
hit = int(
    test_product in
    test_recommendations["Recommended_Product"].values
)

hit_rate = hit / 1

print("Hit Rate:", hit_rate)
print("Hit Rate (%):", hit_rate * 100)

Hit Rate: 1.0
Hit Rate (%): 100.0


In [11]:
# Save recommendation results
recommendation_df.to_csv(
    "day32_recommendation_outputs.csv",
    index=False
)

# Save customer similarity scores
similarity_df.to_csv(
    "day32_similarity_analysis.csv"
)

# Save evaluation result
evaluation_df = pd.DataFrame({
    "Metric": ["Hit Rate"],
    "Value": [hit_rate],
    "Test_Cases": [1]
})

evaluation_df.to_csv(
    "day32_recommendation_evaluation.csv",
    index=False
)

print("Day 32 reports saved successfully!")

Day 32 reports saved successfully!
